# Turbopuffer RAG Training Pipeline

This notebook provides a streamlined 4-step process to generate synthetic QA data and launch a training job using Turbopuffer as the corpus backend:

1. **Point to dataset** - Chunk and upload documents to a Turbopuffer namespace
2. **Create synthetic QA** - Generate question-answer pairs with CgftPipeline
3. **Create env** - Configure the Turbopuffer search environment
4. **Run training job** - Launch the training

## Setup

Install dependencies and configure API credentials.

In [ ]:
# For Google Colab - clone repo and install
# !git clone https://github.com/cgftinc/synthetic-data-prep-lib.git
# %cd synthetic-data-prep-lib
# %pip install -e .[dev] turbopuffer

In [ ]:
# Local development setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

cwd = Path.cwd()
candidate_roots = [cwd, cwd.parent]
repo_root = next((p for p in candidate_roots if (p / "src" / "synthetic_data_prep").exists()), cwd)
src_path = repo_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
from synthetic_data_prep.utils import ensure_safe_python_version

ensure_safe_python_version()

In [ ]:
# Configuration

# Turbopuffer credentials — get your API key at https://turbopuffer.com/
TURBOPUFFER_API_KEY = "[INSERT API KEY]"
TURBOPUFFER_REGION = "aws-us-east-1"
NAMESPACE = "my-docs"

# CGFT API key — create one at https://app.cgft.io/account/api-keys
API_KEY = "[INSERT API KEY]"
BASE_URL = "https://app.cgft.io"

# LLM API credentials — used by CgftPipeline for QA generation, filtering, and refinement
LLM_API_KEY = "[INSERT LLM API KEY]"
LLM_BASE_URL = "https://expt-platform-foundry.openai.azure.com/openai/v1"

# Dataset configuration
# This should point to a local directory containing the documents you want to use.
# Go to docs.cgft.io/rag/synthetic_datagen for sample documents you can use.
DOCS_PATH = "./samples/posthog/docs/"

# Corpus intent used for corpus profiling (summary + example queries)
CORPUS_DESCRIPTION = "Posthog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]

# QA generation configuration (CgftPipeline)
TOTAL_SAMPLES = 100
OUTPUT_DIR = "outputs/tpuf_rag"

# Optional: name for your training experiment
EXPERIMENT_NAME = None  # e.g. "posthog-tpuf-v1"

# Optional: provide Azure OpenAI credentials to store vector embeddings alongside BM25.
# Set EMBED_API_KEY to None to use BM25 full-text search only (no vectors stored).
EMBED_API_KEY = None      # e.g. "4T3rpNzE..."
EMBED_ENDPOINT = None     # e.g. "https://my-resource.cognitiveservices.azure.com/"
EMBED_DEPLOYMENT = None   # e.g. "text-embedding-3-small"
EMBED_API_VERSION = "2024-12-01-preview"

## Step 1: Point to Dataset

Chunk your documents and upload to a Turbopuffer namespace in one line.

> If you already have a populated namespace you want to use, skip this step.

In [ ]:
from synthetic_data_prep.corpus.turbopuffer.source import TpufChunkSource

source = TpufChunkSource(api_key=TURBOPUFFER_API_KEY, namespace=NAMESPACE, region=TURBOPUFFER_REGION)

# Optional: build an embedding function to store vectors alongside BM25.
# Remove this block (or leave EMBED_API_KEY=None above) for BM25-only.
embed_fn = None
if EMBED_API_KEY:
    from openai import AzureOpenAI
    _openai = AzureOpenAI(api_key=EMBED_API_KEY, api_version=EMBED_API_VERSION, azure_endpoint=EMBED_ENDPOINT)
    def embed_fn(texts):
        response = _openai.embeddings.create(model=EMBED_DEPLOYMENT, input=texts)
        return [item.embedding for item in sorted(response.data, key=lambda x: x.index)]

source.populate_from_folder(DOCS_PATH, embed_fn=embed_fn)

## Step 2: Create Synthetic QA

Generate synthetic QA pairs with `CgftPipeline` using the Turbopuffer corpus from Step 1.

In [ ]:
from synthetic_data_prep.qa_generation.cgft_models import (
    CgftPipelineConfig,
    CorpusConfig,
    CorpusContextConfig,
    EntityExtractionLLMConfig,
    FilteringConfig,
    GenerationConfig,
    GroundingLLMFilterConfig,
    LinkerConfig,
    LLMDirectGenerationConfig,
    LLMEnvGenerationConfig,
    LLMGuidedLinkerConfig,
    OutputConfig,
    PlatformConfig,
    RefinementConfig,
    RetrievalLLMFilterConfig,
    TargetsConfig,
    TransformationConfig,
)
from synthetic_data_prep.qa_generation.cgft_pipeline import CgftPipeline

LLM_MODEL = "gpt-5.4"

# --- Batch / concurrency settings ---
# max_concurrent controls how many LLM calls run in parallel within each stage.
# batch_enabled=True  → parallel LLM calls (default, recommended for > ~5 samples)
# batch_enabled=False → sequential calls (useful for debugging a single request)
MAX_CONCURRENT = 8

cfg = CgftPipelineConfig(
    platform=PlatformConfig(api_key=API_KEY, base_url=BASE_URL, llm_api_key=LLM_API_KEY, llm_base_url=LLM_BASE_URL),
    corpus=CorpusConfig(corpus_name=NAMESPACE, min_chunk_chars=400),
    corpus_context=CorpusContextConfig(
        description=CORPUS_DESCRIPTION,
        example_queries=EXAMPLE_QUERIES,
        model=LLM_MODEL,
        num_top_level_samples=4,
        num_random_samples=4,
        min_chunk_chars=400,
        # generate_entity_patterns=True (default): LLM generates entity/code/domain patterns
        # from corpus samples during profiling, improving BM25 chunk linking quality.
        # Set to False to skip the extra LLM call and fall back to metadata-based BM25 queries.
        entity_extraction_llm=EntityExtractionLLMConfig(model=LLM_MODEL)
    ),
    linker=LinkerConfig(
        llm_guided=LLMGuidedLinkerConfig(model=LLM_MODEL),
    ),
    generation=GenerationConfig(
        llm_direct=LLMDirectGenerationConfig(
            model=LLM_MODEL,
            max_concurrent=MAX_CONCURRENT,
            batch_enabled=True,
            show_batch_progress=True,
        ),
        llm_env=LLMEnvGenerationConfig(model=LLM_MODEL),
    ),
    transformation=TransformationConfig(model=LLM_MODEL, validation_model=LLM_MODEL),
    filtering=FilteringConfig(
        retrieval_llm=RetrievalLLMFilterConfig(
            judge_model=LLM_MODEL,
            max_concurrent=MAX_CONCURRENT,
            batch_enabled=True,
        ),
        grounding_llm=GroundingLLMFilterConfig(
            judge_model=LLM_MODEL,
            max_concurrent=MAX_CONCURRENT,
            batch_enabled=True,
        ),
    ),
    refinement=RefinementConfig(model=LLM_MODEL),
    targets=TargetsConfig(total_samples=TOTAL_SAMPLES),
    output=OutputConfig(dir=OUTPUT_DIR, train_jsonl="train.jsonl", eval_jsonl="eval.jsonl"),
)

cfg.resolve_api_keys()
print("generator model:", cfg.generation.llm_direct.model)
print("max_same_seed_attempts_before_reanchor:", cfg.refinement.max_same_seed_attempts_before_reanchor)
print("filter chain:", cfg.filtering.filters)
print("generate_entity_patterns:", cfg.corpus_context.generate_entity_patterns)
print()
print("Batch settings")
print(f"  generation  → batch_enabled={cfg.generation.llm_direct.batch_enabled}, max_concurrent={cfg.generation.llm_direct.max_concurrent}, show_progress={cfg.generation.llm_direct.show_batch_progress}")
print(f"  grounding   → batch_enabled={cfg.filtering.grounding_llm.batch_enabled}, max_concurrent={cfg.filtering.grounding_llm.max_concurrent}")
print(f"  retrieval   → batch_enabled={cfg.filtering.retrieval_llm.batch_enabled}, max_concurrent={cfg.filtering.retrieval_llm.max_concurrent}")

### Batch / parallel processing

With `batch_enabled=True` (the default), LLM calls within each pipeline stage run concurrently up to `max_concurrent`. Stages themselves still run in order:

```
Generator.generate(tasks)
    ├─ prepare all tasks sequentially (linker.link per task)
    └─ batch LLM call  ← parallel, up to max_concurrent

GroundingFilter.evaluate(items)
    ├─ BM25 search sequentially (one per item)
    └─ batch judge calls  ← parallel, up to max_concurrent

RetrievalFilter.evaluate(items)
    ├─ BM25 search sequentially
    ├─ items that hit the overlap pre-gate → verdict immediately (no judge call)
    └─ remaining items → batch judge calls  ← parallel
```

Expected speedup vs sequential (`max_concurrent=1`): **3–6× faster** for batches of ≥ 20 items, limited by LLM latency rather than call count.

To debug a single item, set `batch_enabled=False` in any of the three configs above.

In [ ]:
# Reuse the already-loaded corpus source from Step 1.
import importlib
from pprint import pprint

import synthetic_data_prep.qa_generation.cgft_pipeline as _cgft_pipeline_mod
import synthetic_data_prep.qa_generation.generators.direct_llm as _direct_llm_mod

# Force-refresh generation modules in case notebook state still has stale classes.
importlib.reload(_direct_llm_mod)
importlib.reload(_cgft_pipeline_mod)
CgftPipeline = _cgft_pipeline_mod.CgftPipeline

cgft = CgftPipeline(cfg, source_factory=lambda _cfg: source)
result = cgft.run()

train_data = result["train_dataset"]
eval_data = result["eval_dataset"]
stats = result["stats"]

# Show entity extraction patterns generated during corpus profiling.
entity_extraction = cfg.corpus_context.entity_extraction
if entity_extraction is not None:
    print("Entity extraction patterns (confidence=%s)" % entity_extraction.confidence)
    pprint(
        {
            "entity_names": entity_extraction.entity_names,
            "domain_terms": entity_extraction.domain_terms,
            "code_patterns": entity_extraction.code_patterns,
            "query_templates": entity_extraction.query_templates,
        }
    )
else:
    print("Entity extraction patterns: not generated (generate_entity_patterns=False or profiling disabled)")

print()
print("Run summary")
pprint(
    {
        "raw_candidates_total": stats.get("raw_candidates_total"),
        "passed_total": stats.get("passed_total"),
        "rejected_total": stats.get("rejected_total"),
        "regenerated_total": stats.get("regenerated_total"),
        "round_limit": stats.get("round_limit"),
        "filter_chain": stats.get("filter_chain"),
    }
)

retrieval_stats = stats.get("retrieval_too_easy_filter") or {}
if retrieval_stats:
    print()
    print("Retrieval filter diagnostics")
    pprint(
        {
            "passed": retrieval_stats.get("passed"),
            "needs_refinement": retrieval_stats.get("needs_refinement"),
            "rejected": retrieval_stats.get("rejected"),
            "errors": retrieval_stats.get("errors"),
            "judge_calls": retrieval_stats.get("judge_calls"),
            "judge_answerable_true": retrieval_stats.get("judge_answerable_true"),
            "judge_answerable_false": retrieval_stats.get("judge_answerable_false"),
            "too_easy_total": retrieval_stats.get("too_easy_total"),
            "too_easy_due_to_overlap_pre_gate": retrieval_stats.get("too_easy_due_to_overlap_pre_gate"),
            "too_easy_due_to_judge": retrieval_stats.get("too_easy_due_to_judge"),
            "overlap_threshold_triggered": retrieval_stats.get("overlap_threshold_triggered"),
            "too_easy_overlap_threshold_triggered": retrieval_stats.get("too_easy_overlap_threshold_triggered"),
        }
    )
    print("judge_reason_tags:")
    pprint(retrieval_stats.get("judge_reason_tags", {}))
else:
    print()
    print("Retrieval (too-easy) filter diagnostics are unavailable in result['stats'].")

grounding_stats = stats.get("grounding_filter", {})
if grounding_stats:
    print()
    print("Grounding filter diagnostics")
    pprint(
        {
            "passed": grounding_stats.get("passed"),
            "needs_refinement": grounding_stats.get("needs_refinement"),
            "rejected": grounding_stats.get("rejected"),
            "errors": grounding_stats.get("errors"),
            "judge_answerable_true": grounding_stats.get("judge_answerable_true"),
            "judge_answerable_false": grounding_stats.get("judge_answerable_false"),
            "unsupported_total": grounding_stats.get("unsupported_total"),
        }
    )

transformation_stats = stats.get("transformation", {})
if transformation_stats:
    print()
    print("Transformation diagnostics")
    pprint(transformation_stats)

stats

In [ ]:
for qa_pair in result['train_dataset'][:10]:
    print("Q:", qa_pair['question'])
    print("A:", qa_pair['answer'])

## Step 3: Launch Training

Upload datasets and environment, then launch the training job.

In [ ]:
import synthetic_data_prep
from synthetic_data_prep.envs.tpuf_search_env import TpufSearchEnv
from synthetic_data_prep.trainer.pipeline import train

experiment_id = train(
    env_class=TpufSearchEnv,
    env_args={
        "turbopuffer_api_key": TURBOPUFFER_API_KEY,
        "namespace": NAMESPACE,
        "region": TURBOPUFFER_REGION,
    },
    train_dataset=train_data,
    eval_dataset=eval_data,
    prefix="tpuf-search",
    api_key=API_KEY,
    base_url=BASE_URL,
    local_modules=[synthetic_data_prep],
    pip_dependencies=["turbopuffer"],
    experiment_name=EXPERIMENT_NAME,
)

## Monitoring Training: What to Expect

### Early Training Noise

**Early training**: At the start of a training run, rewards will fluctuate and metrics may be noisy. This is completely normal - the model is still learning basic patterns and its outputs are unstable. Give it some time (usually a few dozen steps) and the signal will clean up and you'll start seeing clear trends.

**Exploration before exploitation**: Ideally, you want to see pass@k climbing first before average reward increases significantly. This means your model is exploring the solution space and learning to solve more prompts (breadth) before it optimizes for higher rewards on those prompts (depth). If average reward shoots up while pass@k stays low, you're likely exploiting a narrow set of easy prompts rather than building robust capabilities.

**Healthy training trajectory**: In a well-configured training run, expect pass@k to increase early as the model learns to solve more distinct prompts. Average reward should follow but lag behind. Eventually both should plateau as the model saturates your training distribution.

**Warning signs**:

- pass@k flat while average reward increases → model is overfitting to a narrow subset of prompts
- both metrics stagnate early → training distribution may be too hard, reward signal too sparse